<a href="https://colab.research.google.com/github/PiaokertheKTH/cb2330-portfolio/blob/main/exercise_part_B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this part, I model a sequencing read using a first-order Markov chain. The first base is described by its marginal probability. Each later base is
conditioned on the preceding base.


I randomly selected one of the three phage genomes and extracted a continuous 40-base read from a random position. The source identity was not used during classification. I calculated the read likelihood under the T4, T7 and M13 first-order Markov models and selected the phage with the largest posterior probability. A fixed random seed was used to make the experiment reproducible.

In [7]:
import random

def markov_likelihood(genome, read):
    """
    Calculate the likelihood of a read under a first-order
    Markov-chain model.
    """

    if len(read) == 0:
        raise ValueError("The read must not be empty.")

    # The first base has no preceding base,so use its ordinary base probability.
    likelihood = p_base(genome, read[0])
    # From the second base onwards, condition each base on the preceding base.
    for i in range(1, len(read)):
        previous_base = read[i - 1]
        current_base = read[i]

        transition_probability = p_next_given(
            genome,
            previous_base,
            current_base
        )

        likelihood = likelihood * transition_probability

    return likelihood

# 1. Randomly select a phage and extract a 40-base read
RANDOM_SEED = 42
READ_LENGTH = 40
random.seed(RANDOM_SEED)
# Randomly choose T4, T7 or M13.
true_index = random.randrange(len(GENOMES))
true_source = NAMES[true_index]
source_genome = GENOMES[true_index]
# Choose a valid random starting position.
start = random.randint(
    0,
    len(source_genome) - READ_LENGTH
)

read = source_genome[start:start + READ_LENGTH]

assert len(read) == READ_LENGTH
print("Random read:", read)
print("Read length:", len(read))
print("Starting position:", start)

# 2. Calculate the likelihood under every phage model
markov_likelihoods = {}

for i in range(len(NAMES)):
    phage_name = NAMES[i]
    phage_genome = GENOMES[i]

    markov_likelihoods[phage_name] = markov_likelihood(
        phage_genome,
        read
    )

# 3. Give the three phages equal prior probabilities
phage_priors = {
    "T4": 1 / 3,
    "T7": 1 / 3,
    "M13": 1 / 3
}
# Check that the prior probabilities sum to 1.
assert abs(sum(phage_priors.values()) - 1.0) < 1e-12

# 4. Convert likelihoods into posterior probabilities
markov_posteriors = posterior(
    markov_likelihoods,
    phage_priors
)

assert abs(sum(markov_posteriors.values()) - 1.0) < 1e-12

# 5. Select the phage with the largest posterior
predicted_source = max(
    markov_posteriors,
    key=markov_posteriors.get
)

# 6. Display the results
print()
print("Phage     likelihood        posterior")

for name in NAMES:
    print(
        f"{name:4s}      "
        f"{markov_likelihoods[name]:.3e}       "
        f"{markov_posteriors[name]:.3f}"
    )

print()
print("Predicted source:", predicted_source)
print("True source:     ", true_source)
print("Correct:         ", predicted_source == true_source)

T4    11000 bases   AATTTTCCTTATTAGGCCGCAAGGGCCTTCATAGTTTTAG...
T7    11000 bases   TCTCACAGTGTACGGACCTAAAGTTCCCCCATAGGGGGTA...
M13    6407 bases   AACGCTACTACTATTAGTAGAATTGATGCCACCTTTTCAG...
Random read: AGCCTTATTCACTGAATGAGCAGCTTTGTTACGTTGATTT
Read length: 40
Starting position: 912


NameError: name 'p_base' is not defined